# SpO2 Spiking CNN — (Spike-Count Output + Weight-Constrained)

In [1]:
!pip install snntorch openpyxl -q

In [2]:
import os, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import snntorch as snn
from snntorch import surrogate

from sklearn.model_selection import KFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score
)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [4]:
folder_path = '/content/drive/My Drive/To Your Path'
save_path   = '/content/drive/My Drive/To Your Path'
os.makedirs(save_path, exist_ok=True)

WINDOW_SIZE   = 100
STRIDE        = 10
BLOCK_SIZE    = 400
NUM_CHANNELS  = 3
NUM_CLASSES   = 3
CLASS_NAMES   = ['Normal', 'Moderate', 'Severe']
N_SUB         = (BLOCK_SIZE - WINDOW_SIZE) // STRIDE + 1

def label_spo2(spo2):
    if spo2 > 92:
        return 0
    elif spo2 > 80:
        return 1
    else:
        return 2

BETA          = 0.80
NUM_STEPS     = 50
SPIKE_GRAD    = surrogate.fast_sigmoid(slope=25)

BATCH_SIZE    = 64
LEARNING_RATE = 5e-4
EPOCHS        = 250
WEIGHT_DECAY  = 5e-4
N_FOLDS       = 5
SPLIT_SEED    = 6

EXCLUDE_SUBJECTS = ['Subject2']

# Hardware weight limit
HW_WT_MAX = sum(1.0 / (2 ** (i + 1)) for i in range(8))  # 0.996
CONV_WEIGHT_CLAMP = 0.4

torch.manual_seed(42)
np.random.seed(42)

print(f'v14g Config (Spike-Count Output + Weight-Constrained):')
print(f'  T={NUM_STEPS}, Epochs={EPOCHS}')
print(f'  Output: argmax(spike_count) — NOT argmax(mem_sum)')
print(f'  Conv weight clamp: ±{CONV_WEIGHT_CLAMP}')

## Load, Prepare, Normalize

In [ ]:
files = sorted(glob.glob(os.path.join(folder_path, '*Subject*.xlsx')))
print(f'Found {len(files)} subject files\n')
all_dfs = []
for f in files:
    df = pd.read_excel(f)
    all_dfs.append(df)
    subj = df['SubjectID'].iloc[0]
    skin = df['skintone'].iloc[0]
    marker = ' <- EXCLUDED' if subj in EXCLUDE_SUBJECTS else ''
    print(f'  {subj}: {len(df)} rows, skintone={skin:.3f}{marker}')

combined = pd.concat(all_dfs, ignore_index=True)
combined = combined[~combined['SubjectID'].isin(EXCLUDE_SUBJECTS)].reset_index(drop=True)
combined['label_3class'] = combined['SpO2_Rad'].apply(label_spo2)

skintone_map = {}
for subj in combined['SubjectID'].unique():
    skintone_map[subj] = combined[combined['SubjectID'] == subj]['skintone'].iloc[0]

In [ ]:
group_data = []
for subj in sorted(combined['SubjectID'].unique()):
    subj_df = combined[combined['SubjectID'] == subj]
    skin = skintone_map[subj]
    for wts in sorted(subj_df['window_start_timestamp'].unique()):
        block = subj_df[subj_df['window_start_timestamp'] == wts].sort_values('sample_idx')
        assert len(block) == BLOCK_SIZE
        red = block['red_win_filtered'].values.astype(np.float32)
        ir  = block['ir_win_filtered'].values.astype(np.float32)
        label = block['label_3class'].iloc[0]
        sub_windows = []
        for start in range(0, BLOCK_SIZE - WINDOW_SIZE + 1, STRIDE):
            end = start + WINDOW_SIZE
            sub_windows.append(np.stack([red[start:end], ir[start:end]]))
        group_data.append({'subject': subj, 'wts': wts, 'label': label,
                           'skintone': skin, 'samples': np.array(sub_windows, dtype=np.float32)})

all_samples = np.concatenate([g['samples'] for g in group_data], axis=0)
global_min = all_samples.min()
global_max = all_samples.max()
for g in group_data:
    g['samples'] = (g['samples'] - global_min) / (global_max - global_min)
del all_samples

all_skins = np.array([g['skintone'] for g in group_data])
skin_min = all_skins.min()
skin_max = all_skins.max()
for g in group_data:
    g['skintone_norm'] = (g['skintone'] - skin_min) / (skin_max - skin_min)

print(f'Groups: {len(group_data)}')

## Train/Test Split

In [ ]:
rng = np.random.RandomState(SPLIT_SEED)
test_group_indices = []
train_pool_indices = []
for subj in sorted(set(g['subject'] for g in group_data)):
    subj_indices = [i for i, g in enumerate(group_data) if g['subject'] == subj]
    n = len(subj_indices)
    n_test = max(1, int(n * 0.1))
    shuffled = rng.permutation(subj_indices)
    test_group_indices.extend(shuffled[:n_test])
    train_pool_indices.extend(shuffled[n_test:])

X_test = np.concatenate([group_data[i]['samples'] for i in test_group_indices], axis=0)
y_test = np.concatenate([np.full(N_SUB, group_data[i]['label']) for i in test_group_indices]).astype(np.int64)
skin_test = np.concatenate([np.full(N_SUB, group_data[i]['skintone_norm']) for i in test_group_indices]).astype(np.float32)

print(f'Test: {len(y_test)} samples, Train pool: {len(train_pool_indices)} groups')

## LUT + Dataset + Models

In [5]:
def build_spike_lut(num_steps):
    lut = np.zeros((num_steps + 1, num_steps), dtype=np.float32)
    for n in range(1, num_steps + 1):
        indices = np.round(np.linspace(0, num_steps - 1, n)).astype(int)
        indices = np.clip(indices, 0, num_steps - 1)
        lut[n, indices] = 1.0
    return torch.tensor(lut, dtype=torch.float32)

class FastSpikeDataset(Dataset):
    def __init__(self, X, skin, y, num_steps, lut, augment=False):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.skin = torch.tensor(skin, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.num_steps = num_steps
        self.lut = lut
        self.augment = augment
    def __len__(self):
        return len(self.X)
    def _augment_ppg(self, ppg):
        if torch.rand(1).item() < 0.5:
            ppg = ppg + torch.randn_like(ppg) * 0.02
        if torch.rand(1).item() < 0.5:
            ppg = ppg * (0.85 + torch.rand(1).item() * 0.30)
        if torch.rand(1).item() < 0.5:
            ppg = torch.roll(ppg, shifts=torch.randint(-10, 11, (1,)).item(), dims=1)
        if torch.rand(1).item() < 0.3:
            ppg[0] = ppg[0] * (0.9 + torch.rand(1).item() * 0.2)
            ppg[1] = ppg[1] * (0.9 + torch.rand(1).item() * 0.2)
        return torch.clamp(ppg, 0.0, 1.0)
    def __getitem__(self, idx):
        ppg = self.X[idx].clone()
        if self.augment:
            ppg = self._augment_ppg(ppg)
        skin_ch = self.skin[idx].expand(100).unsqueeze(0)
        signal_3ch = torch.cat([ppg, skin_ch], dim=0)
        n_spikes = torch.round(signal_3ch * self.num_steps).long()
        n_spikes = torch.clamp(n_spikes, 0, self.num_steps)
        patterns = self.lut[n_spikes.flatten()].reshape(3, 100, self.num_steps)
        spikes = patterns.permute(2, 0, 1)
        return spikes, self.y[idx]

# ── Training model (with BN) — SPIKE COUNT OUTPUT ─────────────────────
# All LIF layers: vth=1.0, beta=0.80, reset='subtract'
# LIF4 spikes normally (no output=True) → classification by spike count
class SpikingCNN_3ch(nn.Module):
    def __init__(self, num_steps):
        super().__init__()
        self.num_steps = num_steps
        self.conv1 = nn.Conv1d(3, 16, kernel_size=5, stride=2, bias=False)
        self.bn1   = nn.BatchNorm1d(16)
        self.lif1  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=5, bias=False)
        self.bn2   = nn.BatchNorm1d(32)
        self.lif2  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc1   = nn.Linear(32 * 9, 64, bias=False)
        self.lif3  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc2   = nn.Linear(64, NUM_CLASSES, bias=False)
        self.lif4  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        spk_rec = []
        for step in range(x.size(0)):
            x_t = x[step]
            cur1 = self.bn1(self.conv1(x_t))
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.bn2(self.conv2(spk1))
            spk2, mem2 = self.lif2(cur2, mem2)
            flat = spk2.view(spk2.size(0), -1)
            spk3, mem3 = self.lif3(self.fc1(flat), mem3)
            spk4, mem4 = self.lif4(self.fc2(spk3), mem4)
            spk_rec.append(spk4)
        return torch.stack(spk_rec).sum(dim=0)  # spike count per class

# ── HW model (BN-folded) — same spike-count output ───────────────────
class SpikingCNN_3ch_HW(nn.Module):
    def __init__(self, num_steps):
        super().__init__()
        self.num_steps = num_steps
        self.conv1 = nn.Conv1d(3, 16, kernel_size=5, stride=2, bias=True)
        self.lif1  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=5, bias=True)
        self.lif2  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc1   = nn.Linear(32 * 9, 64, bias=False)
        self.lif3  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc2   = nn.Linear(64, NUM_CLASSES, bias=False)
        self.lif4  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        spk_rec = []
        for step in range(x.size(0)):
            x_t = x[step]
            cur1 = self.conv1(x_t)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.conv2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            flat = spk2.view(spk2.size(0), -1)
            spk3, mem3 = self.lif3(self.fc1(flat), mem3)
            spk4, mem4 = self.lif4(self.fc2(spk3), mem4)
            spk_rec.append(spk4)
        return torch.stack(spk_rec).sum(dim=0)

print('All classes defined.')
print('  All LIF layers: vth=1.0, beta=0.80, reset=subtract')
print('  Output = sum of SPIKES (spike count), not membrane potential')
print()
print('  Hardware mapping:')
print('    decay_rate = 0.20 (= 1 - beta)')
print('    grow_rate  = 1.0')
print('    vth        = 1.0')
print('    reset_mechanism = 1 (subtract in HW encoding)')

## Training Functions

In [ ]:
def clamp_conv_weights(model, limit=CONV_WEIGHT_CLAMP):
    with torch.no_grad():
        model.conv1.weight.clamp_(-limit, limit)
        model.conv2.weight.clamp_(-limit, limit)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for spikes, targets in loader:
        spikes  = spikes.permute(1, 0, 2, 3).to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        output = model(spikes)  # spike counts (B, 3)
        loss = criterion(output, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        clamp_conv_weights(model)
        total_loss += loss.item() * targets.size(0)
        correct    += (output.argmax(1) == targets).sum().item()
        total      += targets.size(0)
    return total_loss / total, 100.0 * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for spikes, targets in loader:
            spikes  = spikes.permute(1, 0, 2, 3).to(device)
            targets = targets.to(device)
            output = model(spikes)
            loss = criterion(output, targets)
            total_loss += loss.item() * targets.size(0)
            preds = output.argmax(1)
            correct += (preds == targets).sum().item()
            total   += targets.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    return total_loss / total, 100.0 * correct / total, np.array(all_preds), np.array(all_targets)

def collect_inference_data(model, loader):
    model.eval()
    all_inputs, all_outputs, all_targets = [], [], []
    with torch.no_grad():
        for spikes, targets in loader:
            spikes_perm = spikes.permute(1, 0, 2, 3).to(device)
            output = model(spikes_perm)
            all_inputs.append(spikes.cpu())
            all_outputs.append(output.cpu())
            all_targets.append(targets.cpu())
    return {
        'input': torch.cat(all_inputs, dim=0),
        'output': torch.cat(all_outputs, dim=0),
        'actual_targets': torch.cat(all_targets, dim=0),
        'torch_targets': torch.cat(all_outputs, dim=0).argmax(dim=1),
    }

def fold_bn_into_conv(conv, bn):
    gamma = bn.weight.data
    beta = bn.bias.data
    mean = bn.running_mean.data
    var = bn.running_var.data
    eps = bn.eps
    W = conv.weight.data
    b = torch.zeros(W.size(0), device=W.device)
    std = torch.sqrt(var + eps)
    scale = gamma / std
    W_new = W * scale.view(-1, 1, 1)
    b_new = scale * (b - mean) + beta
    return W_new, b_new

def build_hw_model(training_model, num_steps):
    hw = SpikingCNN_3ch_HW(num_steps).to(device)
    W1, b1 = fold_bn_into_conv(training_model.conv1, training_model.bn1)
    hw.conv1.weight.data = W1
    hw.conv1.bias.data = b1
    W2, b2 = fold_bn_into_conv(training_model.conv2, training_model.bn2)
    hw.conv2.weight.data = W2
    hw.conv2.bias.data = b2
    hw.fc1.weight.data = training_model.fc1.weight.data.clone()
    hw.fc2.weight.data = training_model.fc2.weight.data.clone()
    hw.lif1.beta = training_model.lif1.beta
    hw.lif2.beta = training_model.lif2.beta
    hw.lif3.beta = training_model.lif3.beta
    hw.lif4.beta = training_model.lif4.beta
    return hw

print('Functions defined.')

## 5-Fold CV + BN Folding

In [ ]:
SPIKE_LUT = build_spike_lut(NUM_STEPS)
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
train_pool_indices_arr = np.array(train_pool_indices)

test_ds = FastSpikeDataset(X_test, skin_test, y_test, NUM_STEPS, SPIKE_LUT, augment=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

fold_results = []

print(f'Running {N_FOLDS}-fold CV ({EPOCHS} epochs, spike-count output, clamp=±{CONV_WEIGHT_CLAMP})\n')

for fold, (train_idx, val_idx) in enumerate(kf.split(train_pool_indices_arr), 1):
    print(f'\n{"="*65}')
    print(f'FOLD {fold}/{N_FOLDS}')
    print(f'{"="*65}')

    fold_train_groups = train_pool_indices_arr[train_idx]
    fold_val_groups   = train_pool_indices_arr[val_idx]

    X_train = np.concatenate([group_data[i]['samples'] for i in fold_train_groups], axis=0)
    y_train = np.concatenate([np.full(N_SUB, group_data[i]['label']) for i in fold_train_groups]).astype(np.int64)
    skin_train = np.concatenate([np.full(N_SUB, group_data[i]['skintone_norm']) for i in fold_train_groups]).astype(np.float32)

    X_val = np.concatenate([group_data[i]['samples'] for i in fold_val_groups], axis=0)
    y_val = np.concatenate([np.full(N_SUB, group_data[i]['label']) for i in fold_val_groups]).astype(np.int64)
    skin_val = np.concatenate([np.full(N_SUB, group_data[i]['skintone_norm']) for i in fold_val_groups]).astype(np.float32)

    print(f'  Train: {len(y_train)} | Val: {len(y_val)}')

    train_ds = FastSpikeDataset(X_train, skin_train, y_train, NUM_STEPS, SPIKE_LUT, augment=True)
    val_ds   = FastSpikeDataset(X_val, skin_val, y_val, NUM_STEPS, SPIKE_LUT, augment=False)

    class_weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train)
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
    class_counts = np.bincount(y_train, minlength=NUM_CLASSES)
    class_sample_weights = 1.0 / class_counts
    sample_weights = torch.tensor([class_sample_weights[l] for l in y_train], dtype=torch.float32)
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,   num_workers=2)

    torch.manual_seed(42 + fold)
    model = SpikingCNN_3ch(NUM_STEPS).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    warmup_epochs = 5
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_acc = 0
    fold_model_path = os.path.join(save_path, f'fold{fold}_best.pth')

    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), fold_model_path)

        if epoch % 50 == 0 or epoch == 1:
            c1_max = model.conv1.weight.data.abs().max().item()
            c2_max = model.conv2.weight.data.abs().max().item()
            print(f'  Ep {epoch:>3}: Train {tr_acc:.1f}% | Val {va_acc:.1f}% | Best {best_val_acc:.1f}% | '
                  f'c1={c1_max:.3f} c2={c2_max:.3f}')

    elapsed = time.time() - t0

    # Load best and evaluate
    model.load_state_dict(torch.load(fold_model_path, map_location=device))
    _, final_val_acc, _, _ = evaluate(model, val_loader, criterion)
    _, test_acc, test_preds, test_targets = evaluate(model, test_loader, criterion)
    _, final_train_acc, _, _ = evaluate(model, train_loader, criterion)

    # BN fold
    hw_model = build_hw_model(model, NUM_STEPS)
    hw_model.eval()
    _, hw_acc, hw_preds, _ = evaluate(hw_model, test_loader, criterion)
    match = np.array_equal(test_preds, hw_preds)

    c1_folded_max = hw_model.conv1.weight.data.abs().max().item()
    c2_folded_max = hw_model.conv2.weight.data.abs().max().item()

    torch.save(hw_model.state_dict(), os.path.join(save_path, f'fold{fold}_hw.pth'))

    inference_data = collect_inference_data(hw_model, test_loader)
    inference_data['weights'] = {k: v.cpu().clone() for k, v in hw_model.state_dict().items()}
    torch.save(inference_data, os.path.join(save_path, f'inference_hw_fold{fold}.pt'))

    fold_results.append({
        'fold': fold, 'train_acc': final_train_acc, 'val_acc': final_val_acc,
        'test_acc': test_acc, 'hw_acc': hw_acc, 'bn_match': match,
        'c1_folded_max': c1_folded_max, 'c2_folded_max': c2_folded_max,
        'test_preds': test_preds, 'test_targets': test_targets, 'time': elapsed,
    })

    print(f'\n  Fold {fold} done in {elapsed:.0f}s')
    print(f'  Train: {final_train_acc:.1f}% | Val: {final_val_acc:.1f}% | Test: {test_acc:.1f}%')
    print(f'  BN Folding: HW={hw_acc:.1f}% [{"✓" if match else "✗"}]')

## Results

In [ ]:
print(f'\n{"="*75}')
print(f'RESULTS — Spike-Count Output + Weight-Constrained')
print(f'{"="*75}')

test_accs = [r['test_acc'] for r in fold_results]
hw_accs = [r['hw_acc'] for r in fold_results]

print(f'{"Fold":>6} {"Train":>8} {"Val":>8} {"Test":>8} {"HW":>8} {"BN":>4} {"C1max":>8} {"C2max":>8}')
print('-' * 65)
for r in fold_results:
    bn_s = '✓' if r['bn_match'] else '✗'
    c1_s = '✓' if r['c1_folded_max'] <= HW_WT_MAX else '✗'
    c2_s = '✓' if r['c2_folded_max'] <= HW_WT_MAX else '✗'
    print(f'{r["fold"]:>6} {r["train_acc"]:>7.1f}% {r["val_acc"]:>7.1f}% {r["test_acc"]:>7.1f}% '
          f'{r["hw_acc"]:>7.1f}% {bn_s:>4} {r["c1_folded_max"]:>7.4f}{c1_s} {r["c2_folded_max"]:>7.4f}{c2_s}')
print('-' * 65)
print(f'{"Mean":>6} {np.mean([r["train_acc"] for r in fold_results]):>7.1f}% '
      f'{np.mean([r["val_acc"] for r in fold_results]):>7.1f}% '
      f'{np.mean(test_accs):>7.1f}% {np.mean(hw_accs):>7.1f}%')


## Best Fold Details

In [ ]:
best = max(fold_results, key=lambda r: r['test_acc'])
print(f'Best fold: {best["fold"]} (Test: {best["test_acc"]:.1f}%, HW: {best["hw_acc"]:.1f}%)')
print(f'Folded weights: Conv1={best["c1_folded_max"]:.4f}, Conv2={best["c2_folded_max"]:.4f}')
print()
print(classification_report(best['test_targets'], best['test_preds'],
                            target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(best['test_targets'], best['test_preds'])
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'v14g — Fold {best["fold"]} (Acc: {best["test_acc"]:.1f}%, spike-count output)')
plt.tight_layout()
plt.savefig(os.path.join(save_path, 'confusion_matrix_best.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Reload Fold 2 — Use SAVED inference data (correct test split)
# ══════════════════════════════════════════════════════════════════════════
# Loads inference_hw_fold2.pt which was saved during training with
# the CORRECT test split. No re-splitting needed.

import os, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import snntorch as snn
from snntorch import surrogate
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ══════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════
save_path  = '/content/drive/My Drive/To Your Path'
BEST_FOLD  = 2
NUM_STEPS  = 50
BATCH_SIZE = 64
NUM_CLASSES = 3
CLASS_NAMES = ['Normal', 'Moderate', 'Severe']
BETA = 0.80
SPIKE_GRAD = surrogate.fast_sigmoid(slope=25)

# ══════════════════════════════════════════════════════════════════════════
# HW MODEL (spike-count output)
# ══════════════════════════════════════════════════════════════════════════
class SpikingCNN_3ch_HW(nn.Module):
    def __init__(self, num_steps):
        super().__init__()
        self.num_steps = num_steps
        self.conv1 = nn.Conv1d(3, 16, kernel_size=5, stride=2, bias=True)
        self.lif1  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=5, bias=True)
        self.lif2  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc1   = nn.Linear(32 * 9, 64, bias=False)
        self.lif3  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc2   = nn.Linear(64, NUM_CLASSES, bias=False)
        self.lif4  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        spk_rec = []
        for step in range(x.size(0)):
            x_t = x[step]
            cur1 = self.conv1(x_t)
            spk1, mem1 = self.lif1(cur1, mem1)
            cur2 = self.conv2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            flat = spk2.view(spk2.size(0), -1)
            spk3, mem3 = self.lif3(self.fc1(flat), mem3)
            spk4, mem4 = self.lif4(self.fc2(spk3), mem4)
            spk_rec.append(spk4)
        return torch.stack(spk_rec).sum(dim=0)

# ══════════════════════════════════════════════════════════════════════════
# LOAD SAVED INFERENCE DATA (from v14g training run)
# ══════════════════════════════════════════════════════════════════════════
inference_file = os.path.join(save_path, f'inference_hw_fold{BEST_FOLD}.pt')
print(f'Loading saved inference data: {inference_file}')
saved = torch.load(inference_file, map_location='cpu')

test_input = saved['input']
test_targets = saved['actual_targets']
saved_preds = saved['torch_targets']

N_test = test_input.shape[0]
print(f'\nLoaded {N_test} test samples from training run')
print(f'Input shape: {test_input.shape}')
print(f'Test class distribution:')
for cls_id, cls_name in enumerate(CLASS_NAMES):
    n = (test_targets == cls_id).sum().item()
    print(f'  {cls_name}: {n} samples')

# Verify saved predictions
saved_acc = 100.0 * (saved_preds.numpy() == test_targets.numpy()).mean()
print(f'\nSaved predictions accuracy: {saved_acc:.1f}%')

# ══════════════════════════════════════════════════════════════════════════
# LOAD HW MODEL & RE-RUN INFERENCE
# ══════════════════════════════════════════════════════════════════════════
hw_model = SpikingCNN_3ch_HW(NUM_STEPS).to(device)
hw_model.load_state_dict(torch.load(os.path.join(save_path, f'fold{BEST_FOLD}_hw.pth'), map_location=device))
hw_model.eval()
print(f'Loaded HW model: fold{BEST_FOLD}_hw.pth')

class TensorDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets
    def __len__(self):
        return len(self.inputs)
    def __getitem__(self, idx):
        return self.inputs[idx], self.targets[idx]

test_ds = TensorDataset(test_input, test_targets)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

all_inputs = []
all_outputs = []
all_targets = []

with torch.no_grad():
    for spikes, targets in test_loader:
        spikes_perm = spikes.permute(1, 0, 2, 3).to(device)
        output = hw_model(spikes_perm)
        all_inputs.append(spikes.cpu())
        all_outputs.append(output.cpu())
        all_targets.append(targets.cpu())

test_data = torch.cat(all_inputs, dim=0)
output = torch.cat(all_outputs, dim=0)
actual_targets = torch.cat(all_targets, dim=0)
torch_preds = output.argmax(dim=1)

# Verify
rerun_acc = 100.0 * (torch_preds.numpy() == actual_targets.numpy()).mean()
match = torch.equal(torch_preds, saved_preds)
print(f'Re-run accuracy: {rerun_acc:.1f}%')
print(f'Matches saved predictions: {"✓ YES" if match else "✗ NO"}')

# ══════════════════════════════════════════════════════════════════════════
# SAVE FINAL INFERENCE DICTIONARY
# ══════════════════════════════════════════════════════════════════════════
wts = {k: v.cpu().clone() for k, v in hw_model.state_dict().items()}

inference_final = {
    'input': test_data.cpu(),
    'output': output.cpu(),
    'actual_targets': actual_targets.cpu(),
    'torch_targets': torch_preds.cpu(),
    'weights': wts,
}

save_file = os.path.join(save_path, f'inference_all_test_fold{BEST_FOLD}.pt')
torch.save(inference_final, save_file)

# ══════════════════════════════════════════════════════════════════════════
# RESULTS
# ══════════════════════════════════════════════════════════════════════════
preds = torch_preds.numpy()
targets_np = actual_targets.numpy()

print(f'\n{"="*60}')
print(f'v14g Fold {BEST_FOLD} — Spike Count Output ({N_test} samples)')
print(f'{"="*60}')
print(f'  Accuracy: {rerun_acc:.1f}%')
print(f'\n  Per-class accuracy:')
for cls_id, cls_name in enumerate(CLASS_NAMES):
    mask = targets_np == cls_id
    if mask.sum() > 0:
        cls_acc = 100.0 * (preds[mask] == targets_np[mask]).mean()
        print(f'    {cls_name}: {cls_acc:.1f}% ({mask.sum()} samples)')

# Spike count statistics
print(f'\n  Output spike counts (per sample):')
spk_counts = output.numpy()
for cls_id, cls_name in enumerate(CLASS_NAMES):
    print(f'    {cls_name} neuron: mean={spk_counts[:, cls_id].mean():.1f}, '
          f'max={spk_counts[:, cls_id].max():.0f}, min={spk_counts[:, cls_id].min():.0f}')

print(f'\n{classification_report(targets_np, preds, target_names=CLASS_NAMES, zero_division=0)}')

# Confusion matrix
cm = confusion_matrix(targets_np, preds)
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'v14g Fold {BEST_FOLD} — Spike Count (Acc: {rerun_acc:.1f}%)')
plt.tight_layout()
plt.savefig(os.path.join(save_path, f'confusion_matrix_fold{BEST_FOLD}_reload.png'), dpi=150, bbox_inches='tight')
plt.show()

# Weight ranges
print(f'\n  Weight ranges (BN-folded):')
for name, tensor in wts.items():
    if 'weight' in name or 'bias' in name:
        t = tensor.numpy()
        print(f'    {name:<25} min={t.min():.4f}, max={t.max():.4f}, absmax={abs(t).max():.4f}')

print(f'\n  Saved to: {save_file}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Extract MIN/MAX of weights, biases, activations, vmem — ALL layers
# Runs all 465 test samples to find global min/max
# ══════════════════════════════════════════════════════════════════════════

import os, warnings
import numpy as np
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

warnings.filterwarnings('ignore')

save_path = '/content/drive/My Drive/To Your Path'
BEST_FOLD = 2
NUM_STEPS = 50
NUM_CLASSES = 3
BETA = 0.80
SPIKE_GRAD = surrogate.fast_sigmoid(slope=25)

class SpikingCNN_3ch_HW(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(3, 16, kernel_size=5, stride=2, bias=True)
        self.lif1  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.conv2 = nn.Conv1d(16, 32, kernel_size=5, stride=5, bias=True)
        self.lif2  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc1   = nn.Linear(32 * 9, 64, bias=False)
        self.lif3  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')
        self.fc2   = nn.Linear(64, 3, bias=False)
        self.lif4  = snn.Leaky(beta=BETA, threshold=1.0, spike_grad=SPIKE_GRAD, reset_mechanism='subtract')

hw_model = SpikingCNN_3ch_HW()
hw_model.load_state_dict(torch.load(os.path.join(save_path, f'fold{BEST_FOLD}_hw.pth'), map_location='cpu'))
hw_model.eval()

saved = torch.load(os.path.join(save_path, f'inference_hw_fold{BEST_FOLD}.pt'), map_location='cpu')
test_input = saved['input']
N_test = test_input.shape[0]

# ══════════════════════════════════════════════════════════════════════════
# WEIGHTS & BIASES (static — just read from model)
# ══════════════════════════════════════════════════════════════════════════
print(f'{"="*70}')
print(f'WEIGHTS & BIASES')
print(f'{"="*70}')
print(f'{"Layer":<25} {"Shape":<20} {"Min":>10} {"Max":>10} {"AbsMax":>10}')
print('-' * 75)

for name, param in hw_model.named_parameters():
    p = param.detach().numpy()
    print(f'{name:<25} {str(p.shape):<20} {p.min():>10.4f} {p.max():>10.4f} {np.abs(p).max():>10.4f}')

# ══════════════════════════════════════════════════════════════════════════
# ACTIVATIONS & VMEM (dynamic — run all samples)
# ══════════════════════════════════════════════════════════════════════════
print(f'\n{"="*70}')
print(f'ACTIVATIONS & VMEM — Running {N_test} samples...')
print(f'{"="*70}')

# Track min/max for each signal
stats = {
    'conv1_act':  {'min': float('inf'), 'max': float('-inf')},
    'lif1_vmem':  {'min': float('inf'), 'max': float('-inf')},
    'lif1_spk':   {'min': float('inf'), 'max': float('-inf')},
    'conv2_act':  {'min': float('inf'), 'max': float('-inf')},
    'lif2_vmem':  {'min': float('inf'), 'max': float('-inf')},
    'lif2_spk':   {'min': float('inf'), 'max': float('-inf')},
    'fc1_act':    {'min': float('inf'), 'max': float('-inf')},
    'lif3_vmem':  {'min': float('inf'), 'max': float('-inf')},
    'lif3_spk':   {'min': float('inf'), 'max': float('-inf')},
    'fc2_act':    {'min': float('inf'), 'max': float('-inf')},
    'lif4_vmem':  {'min': float('inf'), 'max': float('-inf')},
    'lif4_spk':   {'min': float('inf'), 'max': float('-inf')},
}

# Also track per-channel for conv layers
conv1_act_per_ch = {ch: {'min': float('inf'), 'max': float('-inf')} for ch in range(16)}
conv2_act_per_ch = {ch: {'min': float('inf'), 'max': float('-inf')} for ch in range(32)}
lif1_vmem_per_ch = {ch: {'min': float('inf'), 'max': float('-inf')} for ch in range(16)}
lif2_vmem_per_ch = {ch: {'min': float('inf'), 'max': float('-inf')} for ch in range(32)}
fc1_act_per_neuron = {n: {'min': float('inf'), 'max': float('-inf')} for n in range(64)}
lif3_vmem_per_neuron = {n: {'min': float('inf'), 'max': float('-inf')} for n in range(64)}
fc2_act_per_neuron = {n: {'min': float('inf'), 'max': float('-inf')} for n in range(3)}
lif4_vmem_per_neuron = {n: {'min': float('inf'), 'max': float('-inf')} for n in range(3)}

with torch.no_grad():
    for s in range(N_test):
        sample = test_input[s]

        mem1 = hw_model.lif1.init_leaky()
        mem2 = hw_model.lif2.init_leaky()
        mem3 = hw_model.lif3.init_leaky()
        mem4 = hw_model.lif4.init_leaky()

        for step in range(NUM_STEPS):
            x_t = sample[step].unsqueeze(0)

            # Conv1
            cur1 = hw_model.conv1(x_t)
            v = cur1.numpy()
            stats['conv1_act']['min'] = min(stats['conv1_act']['min'], v.min())
            stats['conv1_act']['max'] = max(stats['conv1_act']['max'], v.max())
            for ch in range(16):
                conv1_act_per_ch[ch]['min'] = min(conv1_act_per_ch[ch]['min'], v[0, ch].min())
                conv1_act_per_ch[ch]['max'] = max(conv1_act_per_ch[ch]['max'], v[0, ch].max())

            # LIF1
            spk1, mem1 = hw_model.lif1(cur1, mem1)
            v = mem1.numpy()
            stats['lif1_vmem']['min'] = min(stats['lif1_vmem']['min'], v.min())
            stats['lif1_vmem']['max'] = max(stats['lif1_vmem']['max'], v.max())
            for ch in range(16):
                lif1_vmem_per_ch[ch]['min'] = min(lif1_vmem_per_ch[ch]['min'], v[0, ch].min())
                lif1_vmem_per_ch[ch]['max'] = max(lif1_vmem_per_ch[ch]['max'], v[0, ch].max())

            # Conv2
            cur2 = hw_model.conv2(spk1)
            v = cur2.numpy()
            stats['conv2_act']['min'] = min(stats['conv2_act']['min'], v.min())
            stats['conv2_act']['max'] = max(stats['conv2_act']['max'], v.max())
            for ch in range(32):
                conv2_act_per_ch[ch]['min'] = min(conv2_act_per_ch[ch]['min'], v[0, ch].min())
                conv2_act_per_ch[ch]['max'] = max(conv2_act_per_ch[ch]['max'], v[0, ch].max())

            # LIF2
            spk2, mem2 = hw_model.lif2(cur2, mem2)
            v = mem2.numpy()
            stats['lif2_vmem']['min'] = min(stats['lif2_vmem']['min'], v.min())
            stats['lif2_vmem']['max'] = max(stats['lif2_vmem']['max'], v.max())
            for ch in range(32):
                lif2_vmem_per_ch[ch]['min'] = min(lif2_vmem_per_ch[ch]['min'], v[0, ch].min())
                lif2_vmem_per_ch[ch]['max'] = max(lif2_vmem_per_ch[ch]['max'], v[0, ch].max())

            # FC1
            flat = spk2.view(1, -1)
            cur3 = hw_model.fc1(flat)
            v = cur3.numpy()
            stats['fc1_act']['min'] = min(stats['fc1_act']['min'], v.min())
            stats['fc1_act']['max'] = max(stats['fc1_act']['max'], v.max())
            for n in range(64):
                fc1_act_per_neuron[n]['min'] = min(fc1_act_per_neuron[n]['min'], v[0, n])
                fc1_act_per_neuron[n]['max'] = max(fc1_act_per_neuron[n]['max'], v[0, n])

            # LIF3
            spk3, mem3 = hw_model.lif3(cur3, mem3)
            v = mem3.numpy()
            stats['lif3_vmem']['min'] = min(stats['lif3_vmem']['min'], v.min())
            stats['lif3_vmem']['max'] = max(stats['lif3_vmem']['max'], v.max())
            for n in range(64):
                lif3_vmem_per_neuron[n]['min'] = min(lif3_vmem_per_neuron[n]['min'], v[0, n])
                lif3_vmem_per_neuron[n]['max'] = max(lif3_vmem_per_neuron[n]['max'], v[0, n])

            # FC2
            cur4 = hw_model.fc2(spk3)
            v = cur4.numpy()
            stats['fc2_act']['min'] = min(stats['fc2_act']['min'], v.min())
            stats['fc2_act']['max'] = max(stats['fc2_act']['max'], v.max())
            for n in range(3):
                fc2_act_per_neuron[n]['min'] = min(fc2_act_per_neuron[n]['min'], v[0, n])
                fc2_act_per_neuron[n]['max'] = max(fc2_act_per_neuron[n]['max'], v[0, n])

            # LIF4
            spk4, mem4 = hw_model.lif4(cur4, mem4)
            v = mem4.numpy()
            stats['lif4_vmem']['min'] = min(stats['lif4_vmem']['min'], v.min())
            stats['lif4_vmem']['max'] = max(stats['lif4_vmem']['max'], v.max())
            for n in range(3):
                lif4_vmem_per_neuron[n]['min'] = min(lif4_vmem_per_neuron[n]['min'], v[0, n])
                lif4_vmem_per_neuron[n]['max'] = max(lif4_vmem_per_neuron[n]['max'], v[0, n])

        if (s + 1) % 100 == 0:
            print(f'  {s+1}/{N_test}')

# ══════════════════════════════════════════════════════════════════════════
# RESULTS — GLOBAL
# ══════════════════════════════════════════════════════════════════════════
HW_MAX = 127.996  # Q7.8 range

print(f'\n{"="*70}')
print(f'GLOBAL MIN/MAX across {N_test} samples × {NUM_STEPS} timesteps')
print(f'Hardware range: ±{HW_MAX:.3f} (Q7.8)')
print(f'{"="*70}')
print(f'{"Signal":<20} {"Min":>10} {"Max":>10} {"AbsMax":>10} {"Overflow?":>10}')
print('-' * 65)

for name, s in stats.items():
    absmax = max(abs(s['min']), abs(s['max']))
    overflow = '⚠ YES' if absmax > HW_MAX else '✓ OK'
    print(f'{name:<20} {s["min"]:>10.4f} {s["max"]:>10.4f} {absmax:>10.4f} {overflow:>10}')

# ══════════════════════════════════════════════════════════════════════════
# RESULTS — PER CHANNEL/NEURON
# ══════════════════════════════════════════════════════════════════════════
print(f'\n{"="*70}')
print(f'CONV1 ACTIVATION — per output channel')
print(f'{"="*70}')
for ch in range(16):
    s = conv1_act_per_ch[ch]
    print(f'  ch{ch:>2}: [{s["min"]:>8.4f}, {s["max"]:>8.4f}]')

print(f'\n{"="*70}')
print(f'LIF1 VMEM — per output channel')
print(f'{"="*70}')
for ch in range(16):
    s = lif1_vmem_per_ch[ch]
    print(f'  ch{ch:>2}: [{s["min"]:>8.4f}, {s["max"]:>8.4f}]')

print(f'\n{"="*70}')
print(f'CONV2 ACTIVATION — per output channel')
print(f'{"="*70}')
for ch in range(32):
    s = conv2_act_per_ch[ch]
    print(f'  ch{ch:>2}: [{s["min"]:>8.4f}, {s["max"]:>8.4f}]')

print(f'\n{"="*70}')
print(f'LIF2 VMEM — per output channel')
print(f'{"="*70}')
for ch in range(32):
    s = lif2_vmem_per_ch[ch]
    print(f'  ch{ch:>2}: [{s["min"]:>8.4f}, {s["max"]:>8.4f}]')

print(f'\n{"="*70}')
print(f'FC1 ACTIVATION — per neuron (showing top 10 largest)')
print(f'{"="*70}')
fc1_absmax = [(n, max(abs(s['min']), abs(s['max']))) for n, s in fc1_act_per_neuron.items()]
fc1_absmax.sort(key=lambda x: x[1], reverse=True)
for n, am in fc1_absmax[:10]:
    s = fc1_act_per_neuron[n]
    print(f'  neuron{n:>2}: [{s["min"]:>8.4f}, {s["max"]:>8.4f}] absmax={am:.4f}')

print(f'\n{"="*70}')
print(f'LIF3 VMEM — per neuron (showing top 10 largest)')
print(f'{"="*70}')
lif3_absmax = [(n, max(abs(s['min']), abs(s['max']))) for n, s in lif3_vmem_per_neuron.items()]
lif3_absmax.sort(key=lambda x: x[1], reverse=True)
for n, am in lif3_absmax[:10]:
    s = lif3_vmem_per_neuron[n]
    print(f'  neuron{n:>2}: [{s["min"]:>8.4f}, {s["max"]:>8.4f}] absmax={am:.4f}')

print(f'\n{"="*70}')
print(f'FC2 ACTIVATION — per output neuron')
print(f'{"="*70}')
CLASS_NAMES = ['Normal', 'Moderate', 'Severe']
for n in range(3):
    s = fc2_act_per_neuron[n]
    print(f'  {CLASS_NAMES[n]:>10} (n{n}): [{s["min"]:>8.4f}, {s["max"]:>8.4f}]')

print(f'\n{"="*70}')
print(f'LIF4 VMEM — per output neuron')
print(f'{"="*70}')
for n in range(3):
    s = lif4_vmem_per_neuron[n]
    print(f'  {CLASS_NAMES[n]:>10} (n{n}): [{s["min"]:>8.4f}, {s["max"]:>8.4f}]')

In [12]:
# Extract conv1 activation, lif1 vmem, lif1 spikes for N samples (out_ch=0, spatial=0)
import torch, os
import numpy as np

save_path = '/content/drive/My Drive/To Your Path'
INPUT_FILE = '/content/drive/My Drive/To Your Path/hw_q7_8/fold1/input/snncore.spikes_input_moderate_0_50.txt'
OUTPUT_DIR = '/content/drive/My Drive/To Your Path/hw_q7_8/fold1/output'
FOLD = 1
OUT_CH = 0
SPATIAL = 0
T = 50
PAD_SAMPLES = 10
N_SAMPLES =  10         # ← change this to however many samples you want
LINES_PER_SAMPLE = T + PAD_SAMPLES

with open(INPUT_FILE) as f:
    all_lines = [l.strip() for l in f if l.strip()]

print(f'Total lines: {len(all_lines)}')
print(f'Max possible samples: {len(all_lines) // LINES_PER_SAMPLE}')

data = torch.load(os.path.join(save_path, f'inference_hw_fold{FOLD}.pt'),
                  map_location=device)
hw_weights = data['weights']

hw = SpikingCNN_3ch_HW(T).to(device)
hw.load_state_dict({k: v.to(device) for k, v in hw_weights.items()})
hw.eval()

all_act = []
all_vmem = []
all_spk = []

for s in range(N_SAMPLES):
    block_start = s * LINES_PER_SAMPLE
    data_start = block_start + PAD_SAMPLES
    sample_lines = all_lines[data_start : data_start + T]

    spike_input = np.zeros((T, 3, 100), dtype=np.float32)
    for t in range(T):
        line = sample_lines[t]
        for ch in range(3):
            for pos in range(100):
                bit_idx = ch * 100 + pos
                char_idx = 299 - bit_idx
                spike_input[t, ch, pos] = int(line[char_idx])

    spike_input = torch.from_numpy(spike_input).to(device)
    x = spike_input.unsqueeze(1)

    act_trace = []
    vmem_trace = []
    spk_trace = []

    with torch.no_grad():
        mem1 = hw.lif1.init_leaky()

        for t in range(T):
            x_t = x[t]
            cur1 = hw.conv1(x_t)
            spk1, mem1 = hw.lif1(cur1, mem1)

            act_trace.append(cur1[0, OUT_CH, SPATIAL].item())
            vmem_trace.append(mem1[0, OUT_CH, SPATIAL].item())
            spk_trace.append(int(spk1[0, OUT_CH, SPATIAL].item()))

    all_act.append(act_trace)
    all_vmem.append(vmem_trace)
    all_spk.append(spk_trace)
    print(f'  Sample {s}: act range=[{min(act_trace):+.3f}, {max(act_trace):+.3f}], spikes={sum(spk_trace)}')

# Save
act_file = os.path.join(OUTPUT_DIR, f'sw_conv1_act_ch{OUT_CH}_sp{SPATIAL}_{N_SAMPLES}samples.txt')
vmem_file = os.path.join(OUTPUT_DIR, f'sw_lif1_vmem_ch{OUT_CH}_sp{SPATIAL}_{N_SAMPLES}samples.txt')
spk_file = os.path.join(OUTPUT_DIR, f'sw_lif1_spk_ch{OUT_CH}_sp{SPATIAL}_{N_SAMPLES}samples.txt')

for fname, traces in [(act_file, all_act), (vmem_file, all_vmem), (spk_file, all_spk)]:
    with open(fname, 'w') as f:
        for s in range(N_SAMPLES):
            for _ in range(PAD_SAMPLES):
                f.write(f'+0.000000\n')
            for t in range(T):
                if 'spk' in fname:
                    f.write(f'{traces[s][t]}\n')
                else:
                    f.write(f'{traces[s][t]:+.6f}\n')

print(f'\nSaved {N_SAMPLES} samples to:')
print(f'  {act_file}')
print(f'  {vmem_file}')
print(f'  {spk_file}')

In [11]:
# Extract conv2 activation, lif2 vmem, lif2 spikes for N samples (out_ch=0, spatial=0)
import torch, os
import numpy as np

save_path = '/content/drive/My Drive/To Your Path'
INPUT_FILE = '/content/drive/My Drive/To Your Path/fold1/input/snncore.spikes_input_moderate_0_50.txt'
OUTPUT_DIR = '/content/drive/My Drive/To Your Path/hw_q7_8/fold1/output'
FOLD = 1
OUT_CH = 0
SPATIAL = 0
T = 50
PAD_SAMPLES = 10
N_SAMPLES = 10          # ← change this to however many samples you want
LINES_PER_SAMPLE = T + PAD_SAMPLES

with open(INPUT_FILE) as f:
    all_lines = [l.strip() for l in f if l.strip()]

print(f'Total lines: {len(all_lines)}')
print(f'Max possible samples: {len(all_lines) // LINES_PER_SAMPLE}')

data = torch.load(os.path.join(save_path, f'inference_hw_fold{FOLD}.pt'),
                  map_location=device)
hw_weights = data['weights']

hw = SpikingCNN_3ch_HW(T).to(device)
hw.load_state_dict({k: v.to(device) for k, v in hw_weights.items()})
hw.eval()

all_act = []
all_vmem = []
all_spk = []

for s in range(N_SAMPLES):
    block_start = s * LINES_PER_SAMPLE
    data_start = block_start + PAD_SAMPLES
    sample_lines = all_lines[data_start : data_start + T]

    spike_input = np.zeros((T, 3, 100), dtype=np.float32)
    for t in range(T):
        line = sample_lines[t]
        for ch in range(3):
            for pos in range(100):
                bit_idx = ch * 100 + pos
                char_idx = 299 - bit_idx
                spike_input[t, ch, pos] = int(line[char_idx])

    spike_input = torch.from_numpy(spike_input).to(device)
    x = spike_input.unsqueeze(1)

    act_trace = []
    vmem_trace = []
    spk_trace = []

    with torch.no_grad():
        mem1 = hw.lif1.init_leaky()
        mem2 = hw.lif2.init_leaky()

        for t in range(T):
            x_t = x[t]
            cur1 = hw.conv1(x_t)
            spk1, mem1 = hw.lif1(cur1, mem1)
            cur2 = hw.conv2(spk1)
            spk2, mem2 = hw.lif2(cur2, mem2)

            act_trace.append(cur2[0, OUT_CH, SPATIAL].item())
            vmem_trace.append(mem2[0, OUT_CH, SPATIAL].item())
            spk_trace.append(int(spk2[0, OUT_CH, SPATIAL].item()))

    all_act.append(act_trace)
    all_vmem.append(vmem_trace)
    all_spk.append(spk_trace)
    print(f'  Sample {s}: act range=[{min(act_trace):+.3f}, {max(act_trace):+.3f}], spikes={sum(spk_trace)}')

# Save
act_file = os.path.join(OUTPUT_DIR, f'sw_conv2_act_ch{OUT_CH}_sp{SPATIAL}_{N_SAMPLES}samples.txt')
vmem_file = os.path.join(OUTPUT_DIR, f'sw_lif2_vmem_ch{OUT_CH}_sp{SPATIAL}_{N_SAMPLES}samples.txt')
spk_file = os.path.join(OUTPUT_DIR, f'sw_lif2_spk_ch{OUT_CH}_sp{SPATIAL}_{N_SAMPLES}samples.txt')

for fname, traces in [(act_file, all_act), (vmem_file, all_vmem), (spk_file, all_spk)]:
    with open(fname, 'w') as f:
        for s in range(N_SAMPLES):
            for _ in range(PAD_SAMPLES):
                f.write(f'+0.000000\n')
            for t in range(T):
                if 'spk' in fname:
                    f.write(f'{traces[s][t]}\n')
                else:
                    f.write(f'{traces[s][t]:+.6f}\n')

print(f'\nSaved {N_SAMPLES} samples to:')
print(f'  {act_file}')
print(f'  {vmem_file}')
print(f'  {spk_file}')

In [69]:
# Extract conv2 per-channel activation: int_activation[out_ch=0][in_ch=0][spatial=0] for N samples
import torch, os
import numpy as np
import torch.nn.functional as F

save_path = '/content/drive/My Drive/To Your Path'
INPUT_FILE = '/content/drive/My Drive/To Your Path/hw_q7_8/fold1/input/snncore.spikes_input_moderate_0_50.txt'
OUTPUT_DIR = '/content/drive/My Drive/To Your Path/hw_q7_8/fold1/output'
FOLD = 1
OUT_CH = 0
IN_CH = 15
SPATIAL = 0
T = 50
PAD_SAMPLES = 10
N_SAMPLES = 1          # ← change this
LINES_PER_SAMPLE = T + PAD_SAMPLES

with open(INPUT_FILE) as f:
    all_lines = [l.strip() for l in f if l.strip()]

data = torch.load(os.path.join(save_path, f'inference_hw_fold{FOLD}.pt'),
                  map_location=device)
hw_weights = data['weights']

hw = SpikingCNN_3ch_HW(T).to(device)
hw.load_state_dict({k: v.to(device) for k, v in hw_weights.items()})
hw.eval()

# Conv2 weight for just out_ch=0, in_ch=0: shape (1, 1, 5)
w_single = hw.conv2.weight[OUT_CH, IN_CH].unsqueeze(0).unsqueeze(0)

all_traces = []
for s in range(N_SAMPLES):
    block_start = s * LINES_PER_SAMPLE
    data_start = block_start + PAD_SAMPLES
    sample_lines = all_lines[data_start : data_start + T]

    spike_input = np.zeros((T, 3, 100), dtype=np.float32)
    for t in range(T):
        line = sample_lines[t]
        for ch in range(3):
            for pos in range(100):
                bit_idx = ch * 100 + pos
                char_idx = 299 - bit_idx
                spike_input[t, ch, pos] = int(line[char_idx])

    spike_input = torch.from_numpy(spike_input).to(device)
    x = spike_input.unsqueeze(1)

    trace = []
    with torch.no_grad():
        mem1 = hw.lif1.init_leaky()

        for t in range(T):
            x_t = x[t]
            cur1 = hw.conv1(x_t)
            spk1, mem1 = hw.lif1(cur1, mem1)

            # Per-channel conv2: only in_ch=IN_CH, no bias
            spk1_single = spk1[0, IN_CH].unsqueeze(0).unsqueeze(0)  # (1, 1, 48)
            per_ch = F.conv1d(spk1_single, w_single, bias=None, stride=5)
            trace.append(per_ch[0, 0, SPATIAL].item())

    all_traces.append(trace)
    print(f'  Sample {s}: range=[{min(trace):+.3f}, {max(trace):+.3f}]')

# Save
out_file = os.path.join(OUTPUT_DIR, f'sw_conv2_int_act_out{OUT_CH}_in{IN_CH}_sp{SPATIAL}_{N_SAMPLES}samples.txt')
with open(out_file, 'w') as f:
    for s in range(N_SAMPLES):
        for _ in range(PAD_SAMPLES):
            f.write(f'+0.000000\n')
        for t in range(T):
            f.write(f'{all_traces[s][t]:+.6f}\n')

print(f'\nSaved to {out_file}')
print(f'\nConv2 weights [out_ch={OUT_CH}, in_ch={IN_CH}]:')
for k in range(5):
    print(f'  w[{k}] = {hw.conv2.weight[OUT_CH, IN_CH, k].item():+.6f}')